# Exercise 2.4.12 — complete `LoraHooks`

> Part of [Delta Drills](https://delta-drills.vercel.app) ARENA practice. When the test cell passes, your completion is reported back to your account automatically.

**Section:** `2.4 RLHF`  
**Notebook:** `2.4_RLHF_exercises.ipynb`  
**Return to Delta Drills:** [https://delta-drills.vercel.app/?arena_exercise=2.4.12](https://delta-drills.vercel.app/?arena_exercise=2.4.12)


# [2.4] - RLHF (exercises)

> **ARENA [Streamlit Page](https://arena-chapter2-rl.streamlit.app/04_[2.4]_RLHF)**
>
> **Colab: [exercises](https://colab.research.google.com/github/callummcdougall/ARENA_3.0/blob/main/chapter2_rl/exercises/part4_rlhf/2.4_RLHF_exercises.ipynb?t=20260303) | [solutions](https://colab.research.google.com/github/callummcdougall/ARENA_3.0/blob/main/chapter2_rl/exercises/part4_rlhf/2.4_RLHF_solutions.ipynb?t=20260303)**

Please send any problems / bugs on the `#errata` channel in the [Slack group](https://join.slack.com/t/arena-uk/shared_invite/zt-3afdmdhye-Mdb3Sv~ss_V_mEaXEbkABA), and ask any questions on the dedicated channels for this chapter of material.

You can collapse each section so only the headers are visible, by clicking the arrow symbol on the left hand side of the markdown header cells.

Links to all other chapters: [(0) Fundamentals](https://arena-chapter0-fundamentals.streamlit.app/), [(1) Transformer Interpretability](https://arena-chapter1-transformer-interp.streamlit.app/), [(2) RL](https://arena-chapter2-rl.streamlit.app/).

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/headers/header-24.png" width="350">

# Introduction

This section is designed to take you through a full implementation of RLHF (Reinforcement Learning from Human Feedback). Much of this follows on directly from the PPO implementation from yesterday, with only a few minor adjustments and new concepts. You'll (hopefully) be pleased to learn that we're disposing of OpenAI's gym environment for this final day of exercises, and instead going back to our week 1 roots with TransformerLens!

We'll start by discussing how the RL setting we've used for tasks like CartPole and Atari fits into the world of autoregressive transformer language models. We'll then go through standard parts of the PPO setup (e.g. objective function, memory buffer, rollout and learning phases) and show how to adapt them for our transformer. Finally, we'll put everything together into a `RLHFTrainer` class, and perform RLHF on our transformer!

> **Note - these exercises assume you're running on an A100 (either a virtual machine or Colab Pro+).** If you're running on machine with much less VRAM (<24GB), we recommend setting `LOW_GPU_MEM = True` below. This will switch the model to RLHF from `"gpt2-medium"` to `"gpt2-small"`,
as well as adjust some other parameters like the batch size, the number of tokens generated, and some hyperparameters.

For a lecture on the material today, which provides some high-level understanding before you dive into the material, watch the video below:

<iframe width="540" height="304" src="https://www.youtube.com/embed/wW__XFKIESc" frameborder="0" allow="accelerometer; autoplay; encrypted-media; gyroscope; picture-in-picture" allowfullscreen></iframe>

## Content & Learning Objectives

### 1️⃣ RLHF on transformer language models

Most of the exercises today build towards the implementation of the `RLHFTrainer` class, similar to how DQN and PPO have worked these last few days.

> ##### Learning Objectives
>
> - Understand how the RL agent / action / environment paradigm works in the context of autoregressive transformer models
> - Understand how the RLHF algorithm works, and how it fits on top of PPO
> - Learn about value heads, and how they can be used to turn transformers into actor & critic networks with shared architectures
> - Write a full RLHF training loop, and use it to train your transformer with the "maximize output of periods" reward function
> - Observe and understand the instances of mode collapse that occur when training with this reward function
> - Experiment with different reward functions & training hyperparameters

### 2️⃣ LoRA

> ##### Learning Objectives
>
> - Understand the mechanism behind Low-Rank Adaptors, and how they allow for fine-tuning with less resources.
> - Implement LoRA in a transformer model.
> - Fine-tune larger models that would otherwise take too much VRAM to be possible.

### 3️⃣ GRPO LoRA

GRPO is a variant of PPO specialised for doing RLHF on LLMs. It forgoes the critic, and uses the average reward over many rollouts as a baseline instead.

> ##### Learning Objectives
>
> - Understand and implement GRPO
> - Use GRPO + LoRA together to finetune a model.

### ☆ Bonus

This section offers some suggested ways to extend the core RLHF exercises.

> ##### Learning Objectives
>  
> - Improve your RLHF implementation via techniques like differential learning rates, frozen layers, or adaptive KL penalties
> - Perform some exploratory mechanistic interpretability on RLHF'd models
> - Learn about the trlX library, which is designed to train transformers via RLHF in a way which abstracts away many of the low-level details

## Reading

- [Illustrating Reinforcement Learning from Human Feedback (RLHF)](https://huggingface.co/blog/rlhf) (~10 minutes)
    - An accessible and mostly non-technical introduction to RLHF, which discusses it in context of the full pipeline for training autoregressive transformer language models (starting with pretraining, which is what we did in the first day of last week).
- [RLHF+ChatGPT: What you must know](https://www.youtube.com/watch?v=PBH2nImUM5c) (~5 minutes)
    - The first half of this video provides a high-level overview of RLHF, discussing things like mode collapse, and relates this to the [shoggoth meme](https://i.kym-cdn.com/photos/images/original/002/546/572/bd3.png) that many of you have likely seen!
- [DeepSeekMath: Pushing the Limits of Mathematical Reasoning in Open Language Models](https://arxiv.org/pdf/2402.03300) (~20 minutes)
    - Save reading this now until you get to the section for GRPO, and skim as required.

## Setup code

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

chapter = "chapter2_rl"
repo = "ARENA_3.0"
branch = "main"

# Install dependencies
try:
    import jaxtyping
except:
    %pip install transformer_lens jaxtyping eindex-callum wandb

# Get root directory, handling 3 different cases: (1) Colab, (2) notebook not in ARENA repo, (3) notebook in ARENA repo
root = (
    "/content"
    if IN_COLAB
    else "/root"
    if repo not in os.getcwd()
    else str(next(p for p in Path.cwd().parents if p.name == repo))
)

if Path(root).exists() and not Path(f"{root}/{chapter}").exists():
    if not IN_COLAB:
        !sudo apt-get install unzip
        %pip install jupyter ipython --upgrade

    if not os.path.exists(f"{root}/{chapter}"):
        !wget -P {root} https://github.com/callummcdougall/ARENA_3.0/archive/refs/heads/{branch}.zip
        !unzip {root}/{branch}.zip '{repo}-{branch}/{chapter}/exercises/*' -d {root}
        !mv {root}/{repo}-{branch}/{chapter} {root}/{chapter}
        !rm {root}/{branch}.zip
        !rmdir {root}/{repo}-{branch}


if f"{root}/{chapter}/exercises" not in sys.path:
    sys.path.append(f"{root}/{chapter}/exercises")

os.chdir(f"{root}/{chapter}/exercises")

In [ ]:
import os
import sys
import time
from dataclasses import dataclass
from functools import partial
from pathlib import Path
from typing import Callable, Literal

import einops
import numpy as np
import torch as t
import torch.nn as nn
import wandb
from eindex import eindex
from jaxtyping import Float, Int
from rich import print as rprint
from rich.table import Table
from tabulate import tabulate
from torch import Tensor
from tqdm import tqdm
from transformer_lens import HookedTransformer, HookedTransformerConfig
from transformer_lens.hook_points import HookPoint

# Make sure exercises are in the path
chapter = "chapter2_rl"
section = "part4_rlhf"
root_dir = next(p for p in Path.cwd().parents if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

from part4_rlhf import tests, tests_lora  # , tl_ext

device = t.device("mps" if t.backends.mps.is_available() else "cuda" if t.cuda.is_available() else "cpu")


MAIN = __name__ == "__main__"

# 1️⃣ RLHF on transformer language models

> ##### Learning Objectives
>
> - Understand how the RL agent / action / environment paradigm works in the context of autoregressive transformer models
> - Understand how the RLHF algorithm works, and how it fits on top of PPO
> - Learn about value heads, and how they can be used to turn transformers into actor & critic networks with shared architectures
> - Write a full RLHF training loop, and use it to train your transformer with the "maximize output of periods" reward function
> - Observe and understand the instances of mode collapse that occur when training with this reward function
> - Experiment with different reward functions & training hyperparameters

## The "transformer environment"

We'll start by discussing how we apply the reinforcement learning framework of states/actions/rewards to the setting of autoregressive language modelling. Lots of our intuitions should carry over from yesterday, it's just some of the details that have changed!

### States, actions and episodes

Our actor is an autoregressive language model. The actions $a_t$ are the tokens generated by the model (i.e. the action space is the model's vocabulary). The states $s_t$ are **the entire sequence up to that point** (not just the most recent token). In other words, given a state $s_t$ (sequence) and action $a_t$ (token generation), our new state is the concatenation which we'll denote as $s_{t+1} = [s_t \; a_t]$. For every timestep before the end of the episode, the reward is zero, and for the final timestep, the reward is given by the reward function, given the entire sequence $r_T = R(s_T)$.

Each episode is a fixed length (i.e. all our sampled outputs will have the same number of tokens generated from them). Each episode starts with an initial "prefix prompt", which is chosen before the start of training. This means that discoutning would only scale the final reward by a fixed constant, and so we don't need to worry about it here.

### Rewards and value functions

The reward $r_T$ is a function of the sequence $s_T$. Sometimes it will be a very simple function like the sum of periods `.` in the sequence, other times it'll get a bit more complicated (e.g. using a text classification model to estimate the sentiment of a sequence - we'll do this later!).

In our case, we'll only evaluate the reward at the end of the episode. This means we don't really have a concept of discount factors here - the reward only comes once, and as soon as it comes our episode terminates.

The value function $V(s_t)$ is an estimate of the expected sum of future rewards (up to the end of the episode), which in this case means it's an estimate of what the reward $r_T$ will be once we get to the end of the sequence. We'll be adding a value head to our transformer model to estimate this value function (more on this later).

> Note - a key part of RLHF is the actual gathering of and learning from human feedback, in order to train the reward function. We're not going to be doing that here, instead we'll be working with a fixed reward function. This means our implementation today is a lot more like classical reinforcement learning, and we'll be able to structure it in a way which is very similar to yesterday's PPO implementation.

### ~~Generalized~~ Advantage Estimation

We won't be using the GAE formula today for computing advantages, we'll just be directly computing it via $A(s_t, a_t) = Q(s_t, a_t) - V(s_t)$, where $a_t$ is the action which was actually taken and $Q(s_t, a_t)$ is the critic's estimate of the value function at this new state $s_{t+1} = [s_t \; a_t]$.

We can get away with this because our setup has pretty low variance when it comes to the advantage of particular actions. GAE is most helpful when it reduces variance in the advantage estimation (it does this at the cost of introducing more bias from including future value function estimates), and so it's especially useful when our environment is one with high variability when the advantage (and optimal policy) changes significantly between steps. But this doesn't really apply to us, since every action just adds a single token onto our sequence.

That said, you're welcome to experiment with the setup and try to use GAE instead! This is suggested as a bonus exercise at the end.

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/main/misc/transformer-rl-state.png" width="700">

## RLHF Setup

With this context in mind, we're now ready to look at the full RLHF setup we'll be using:

<img src="https://pbs.twimg.com/media/FkLOrrPWYAAiFLF.jpg:large" width="700">

Our autoregressive transformer model (we'll be using GPT2-Small) is the actor, and its value head will play the role of the critic. We follow the standard PPO setup:

- In **rollout phase**, the actor generates a bunch of sequences all starting from the prefix prompt. We compute advantage estimates using the critic network (value head) and store the experiences in memory.
- In **learning phase**, we sample from these generated experiences (i.e. from a bunch of generated sequences of different lengths, some of which might be prefixes of each other). We compute our objective function (which is the sum of the same 3 terms as yesterday) and perform a gradient step wrt it.

The only new element is the **KL prediction shift penalty**. This is a penalty we add to our overall loss function to stop the transformer from diverging too much from its initial distribution. We want to make our transformer maximize reward, but not in a way which causes it to become completely incoherent!

Note that we compute $D_{KL}(\pi_{PPO} || \pi_{base})$, not the other way around. This is because we want to penalize our new model for generating outputs which would be **extremely unlikely under the old model**, i.e. when $\pi_{PPO}$ is high and $\pi_{base}$ is low. We generally want to focus our model's output into a more concentrated version of the distribution it already has. For example in RLHF, we want to keep a low probability on completely incoherent behaviour which the original model would never have generated. But on the other hand, it's clearly fine for there to be some behaviours (e.g. offensive hate speech) which have a nontrivial probability in our base model but near-zero probability in our new model - in fact this is often desireable! For more on the intuition behind this orientation of the distributions in KL divergence, see [this post](https://www.lesswrong.com/posts/no5jDTut5Byjqb4j5/six-and-a-half-intuitions-for-kl-divergence).

<!-- An alternative perspective can be found from [this post](https://www.lesswrong.com/posts/no5jDTut5Byjqb4j5/six-and-a-half-intuitions-for-kl-divergence) - the KL divergence $D_{KL}(P || Q)$ is large when the observations $P$ give you a lot of evidence that your hypothesis $Q$ is false. We want to make sure that the original (probably coherent and sensible) model $Q$ is still a good approximation for how $P$ behaves, i.e. it shouldn't be too obvious when we observe the outputs of $P$ that they've been generated by a different model. -->

<details>
<summary>KL divergence v.s. reverse KL divergence</summary>
Assume $P$ is the true distribution, and $Q$ is the distribution we're trying to fit to $P$.

* $D_{KL}(P || Q) = \sum_x P(x) \log \frac{P(x)}{Q(x)}$ blows up when $Q(x)$ is zero and $P(x)$ is positive, so we would expect that $Q$
tries to "cover" $P$ anywhere where $P(x)$ is positive. This means that minimizing $D_{KL}(\pi_{base} || \pi_{PPO})$ will cause our model to be able to do everything the base model can do, plus it can also do things out-of-distribution for the base model, which is undesirable.

* $D_{KL}(Q || P) = \sum_x Q(x) \log \frac{Q(x)}{P(x)}$ blows up when $P(x)$ is zero and $Q(x)$ is positive, so $Q$ should never assign
any probability mass to something that $P$ doesn't ($P$ "covers" $Q$), but $Q$ will instead try to cover a subset of $P$ that it fits the best.

This can be illustrated with an example. Let $P$ be a mixture of two Gaussians, and $Q \sim \mathcal{N}(\mu, \sigma^2)$ be a unimodal Gaussian (blue) parameterized by $\mu$ and $\sigma^2$. We learn parameters $\mu,\sigma^2$ that minimize both $D_{KL}(P || Q)$ and $D_{KL}(Q || P)$, and draw the resulting distribution $Q$ (here in blue), showing the expected behaviour.

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/refs/heads/main/img/kl_diff.png" width="700">


</details>

### Summary
 
Since we're using a fixed reward function rather than training it from human feedback, our RLHF implementation looks very similar to yesterday's PPO implementation. The differences are summarized in the table below:

| |  PPO (general) | RLHF |   
|---|---|---|
| **States** | Contains partial knowledge of our environment | Sequence of tokens up to this point (and the model's internal state representation of that sequence) |
| **Actions** | Something our agent can do to change its state | Generating a new token, taking us to state $s_{t+1} = [s_t \; a_t]$ |
| **Rewards** | A function of the state, which is computed after each new state is reached | A function of the sequence, can be computed after each new token but we'll just compute it once at the end of the sequence |
| **Multiple steps in parallel?** | Yes, we used `SyncVectorEnv` to parallelize the rollout phase | Yes, we'll pass batches of sequences into the transformer model, generating multiple new tokens at once |
| **Actor & critic networks** | Architectures can be shared (e.g. for Atari) or disjoint (e.g. for CartPole) | Actor is a transformer model, critic is a value head (so most architecture is shared) |
| **Advantage estimation** | Use GAE with discount factor $\lambda$ | Often uses GAE, but we'll just use simple next-step difference $V(s_{t+1}) - V(s_t)$ |
| **Anything extra?** |  | KL penalty on the new policy wrt the baseline policy |

## RLHF training args

Now that you have a rough idea of how our implementation differs from PPO, we'll give you the `RLHFArgs` class and highlight the differences between this and the `PPOArgs` class from yesterday (mostly it's quite similar).

- We're now using `total_phases` to control how long our training lasts for, rather than using `total_timesteps`. This makes more sense for us, because the total number of timesteps (= number of actions we take = number of tokens we generate) will vary depending on the length of the sequences we generate.
- We've removed the arguments `gamma` and `gae_lambda` for computing the advantage function, since as discussed we'll be computing the advantage in a simpler and more direct way (you'll do this in the next exercise).
- We've added the following arguments related to the base model & text sampling:
    - `base_model`, for specifying different base models (default is `"gpt2-small"`)
    - `gen_len`, the length of the sequences we generate.
    - `temperature` and `top_k`, for controlling the sampling temperature of our sequences.
    - `prefix`, the string we use to generate all samples.
- As well as the following extra RLHF-specific arguments:
    - `kl_coef`, for controlling the strength of the KL prediction shift penalty.
    - `reward_fn`, for the reward function we use.
    - `normalize_reward`, for whether we normalize the reward (this won't always be necessary).
- We've also added two learning rates, since it makes sense to have a different learning rate for our value head and the rest of the model (more on this later!).

In [ ]:
# Set default parameters for low GPU memory usage, change if you have more GPU memory

LOW_GPU_MEM = True
BASE_MODEL = "gpt2-small" if LOW_GPU_MEM else "gpt2-medium"
RUN_BASE_RLHF = True

In [ ]:
@dataclass
class RLHFArgs:
    # Basic / global
    seed: int = 1

    # Wandb / logging
    use_wandb: bool = False
    wandb_project_name: str = "RLHF"
    wandb_entity: str | None = None

    # Duration of different phases
    total_phases: int = 100
    batch_size: int = 128
    num_minibatches: int = 4
    batches_per_learning_phase: int = 2

    # Optimization hyperparameters
    base_lr: float = 2e-5
    head_lr: float = 5e-4
    max_grad_norm: float = 1.0
    warmup_steps: int = 20
    final_scale: float = 0.1

    # Computing other PPO loss functions
    clip_coef: float = 0.2
    vf_coef: float = 0.15
    ent_coef: float = 0.001

    # Base model & sampling arguments
    base_model: str = BASE_MODEL
    gen_len: int = 30
    temperature: float = 1.0
    top_k: int = 10
    prefix: str = "This is"
    prepend_bos: bool = True

    # RLHF-specific arguments
    kl_coef: float = 2.5
    reward_fn: Callable = lambda x: 0.0
    normalize_reward: bool = True

    def __post_init__(self):
        assert self.total_phases > self.warmup_steps, "total_phases must be greater than warmup_steps"
        assert self.batch_size % self.num_minibatches == 0, "batch_size should be divisible by num_minibatches"
        self.minibatch_size = self.batch_size // self.num_minibatches

## Value head

If you worked on the Atari exercises yesterday, then you'l be used to the idea of having shared architecture between our policy and value networks. Intuitively, this is because both networks need to learn some kind of high-level encoding of the important variables in the environment - they just do different things with this encoding.

This leads to the idea of a **value head**. A value head is basically just a simple classifier model which we stick to one of the policy network's internal activations. You can think of this as a kind of feature extraction. When it comes to transformer models, we usually attach our value head to **the value of the residual stream at the very last layer, after layernorm but before unembedding**. Recall the key idea of **residual stream as output accumulation** - by the very last layer, it contains the most context about the overall sequence.\*

\*Technically this might not always be true, since there is some evidence that components of a transformer erase information in order to write different information to the residual stream. However, in practice we usually find that the residual stream at the last layer is the most useful for downstream tasks.

How do we implement this? Before you read further down, try to think about how you might implement this yourself, i.e. how you could extend the functionality of your `HookedTransformer` model by adding a value head, without completely rewriting the `HookedTransformer` architecture.

<details>
<summary>Hint</summary>

Think about using hook functions.

</details>

<details>
<summary>Answer</summary>

One method would be to directly edit the model by replacing its modules with different ones. But this is a bit awkward, because we have to also change modules which are downstream of the value head to make sure that they're only taking the residual stream as input (not the value head's output), etc.

A different method, which is what we'll be using in these exercises, is to use **hook functions**. We can attach a hook function to the residual stream at the final layer, and have it apply our value head to the residual stream values & store the output externally. Then we can use `model.run_with_hooks` to get our logits like normal, and fetch our value estimate from the external storage object.

We're used to using hook functions during inference mode to perform causal interventions or compute statistical functions of our activations, but they can also be used during training mode to perform computations which are part of the autograd's computational graph.

</details>

## Connect to Delta Drills

Paste your Delta Drills auth token below so this exercise can report its completion back to your account.
You can copy the token from your Delta Drills account page.


In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_EXERCISE_ID = "2.4.12"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"


### Prior-exercise solutions (auto-imported)

These were imported from ARENA's reference `solutions.py` so you can jump straight into this exercise without having implemented every predecessor. Re-implement them yourself if you'd rather build top-to-bottom.


In [ ]:
from part4_rlhf.solutions import HookedTransformerWithValueHead, reward_fn_char_count, normalize_reward, normalize_reward, compute_advantages, calc_kl_penalty, calc_entropy_bonus, get_logprobs, get_optimizer, RLHFTrainer, reward_fn_sentiment_imdb, Lora


### Exercise - complete `LoraHooks`

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Importance: 🔵🔵🔵🔵🔵
> 
> You should spend up to 25-30 minutes on this exercise.
> ```
Now you'll implement the `LoraHooks` class. This class should define LoRA modules for the linear projections inside the attention layer of the transformer. 

The following methods have been implemented for you:

* The hook function `store_hook_attn_normalized` should cache the input to querys, keys, and values.
* The hook function `store_hook_z` should cache the input to $W_O$.
* The method `list_fwd_hooks` should return a list of hook_point names and functions to call for the forward pass of the model using LoRA.

You should
* Define `self.lora_q`, `self.lora_k`, `self.lora_v`, `self.lora_o` of appropriate sizes.
* Implement the `lora_hook_qkv` method to apply the LoRA modules to the input to the attention layer.
   - This function should check the hook location (`hook.name`) and apply the appropriate LoRA modules to the appropriate input.
   - Note that `normalized : Float[Tensor, "batch pos d_model"]` is the input to the attention layer, which we need to repeat for each head before passing to the LoRA modules.
* Implement the `lora_hook_out` method to apply the LoRA modules to the output of the attention layer.
   - Note that `transformer_lens` doesn't hook the output of $W_O$ *before* the heads are summed over. Lucky for us, LoRA performs a linear operation, so we can sum the output of the LoRA model for $W_0$ over each head, and then add to the output!
   For example, for two heads:
   
   $$ 
   (\tilde{W}^1_O + \tilde{W}^2_O)(x) = x(W^1_O + A^1 B^1 + W^2_O + A^2 B^2) = x(W^1_O + W^2_O) + ((xA^1)B^1 + (xA^2) B^2)
   $$

<details>
<summary>What's the deal with <code>n_qo_heads</code> and <code>n_kv_heads</code>?</summary>

**TL;DR:** All you need to know is use `n_qo_heads` for the number of query heads and output heads, and `n_kv_heads` for the number of key and value heads.

For `gpt2`, we have the same number of heads in each layer, and each head has linear projections 
* $W_Q : (n_{heads},d_{model}, d_{head})$
* $W_K : (n_{heads},d_{model}, d_{head})$
* $W_V : (n_{heads},d_{model}, d_{head})$
* $W_O : (n_{heads}, d_{head}, d_{model})$

This turns out to be costly on memory when using KV-caching, so a solution proposed was [**grouped-query attention**](https://arxiv.org/pdf/2305.13245),
where there are fewer key and value heads than query heads. The query heads are put into groups, and each group shares the same
key and value head.

<img src="https://raw.githubusercontent.com/info-arena/ARENA_img/f0d17ee6b9eb89dd72b746c43852a2dcb1245733/img/grouped_query_attention.png" width="960|">

We can see this in the family of Llama models which makes use of this technique.

```python
model = transformer_lens.HookedTransformer.from_pretrained("meta-llama/Llama-3.2-1B")
print(f"{model.cfg.n_heads=}")
print(f"{model.cfg.n_key_value_heads=}")
print(f"Group size: {model.cfg.n_heads // model.cfg.n_key_value_heads}")
print(f"{(model.W_K[0][:4] == model.W_K[0][0]).all()=}") 
```

<pre style="white-space:pre;overflow-x:auto;line-height:normal;font-family:Menlo,'DejaVu Sans Mono',consolas,'Courier New',monospace">
Loaded pretrained model meta-llama/Llama-3.2-1B into HookedTransformer
model.cfg.n_heads=32
model.cfg.n_key_value_heads=8
Group size: 4
(model.W_K[0][:4] == model.W_K[0][0]).all()=tensor(True, device='cuda:0')
</pre>

Note that transformer lens presents `W_K` and `W_V` as the same shape as `W_Q`, but this is a lie, they are only a repeated view of the true key and value matricies, which are smaller. We can see this by looking inside the attention layer: `_W_K` is the *real* weights, and `W_K` is a repeated view of it.

```python
print(f"{model.blocks[0].attn._W_K.shape=}")
print(f"{model.blocks[0].attn.W_K.shape=}")
```

<pre style="white-space:pre;overflow-x:auto;line-height:normal;font-family:Menlo,'DejaVu Sans Mono',consolas,'Courier New',monospace">
model.blocks[0].attn._W_K.shape=torch.Size([8, 2048, 64])
model.blocks[0].attn.W_K.shape=torch.Size([32, 2048, 64])
</pre>


</details>

In [ ]:
class LoraHooks(nn.Module):
    """
    Defines the LoRA hooks needed for the Attention Layers of the transformer.
    (Could be modified to add LoRA to the MLP layers)
    """

    lora_q: Lora
    lora_k: Lora
    lora_v: Lora
    lora_o: Lora
    cache_qkv_in: Float[Tensor, "batch pos d_model"] = None
    cache_z: Float[Tensor, "batch pos n_heads d_head"] = None

    def __init__(
        self,
        layer_idx: int,
        cfg: HookedTransformerConfig,
        lora_alpha: float = 32,
        rank: int = 4,
        dtype: t.dtype = None,
    ):
        super().__init__()
        self.layer_idx = layer_idx
        self.rank = rank
        self.lora_alpha = lora_alpha
        self.dtype = dtype

        self.n_qo_heads = n_qo_heads = cfg.n_heads
        self.n_kv_heads = n_kv_heads = cfg.n_key_value_heads if cfg.n_key_value_heads is not None else cfg.n_heads
        d_model, d_head = cfg.d_model, cfg.d_head

        raise NotImplementedError()

    def store_hook_attn_normalized(self, normalized: Float[Tensor, "batch pos d_model"], hook: HookPoint) -> None:
        """
        Cache the input to query/key/value.
        """
        self.cache_qkv_in = normalized

    def store_hook_z(self, z: Float[Tensor, "batch pos n_heads d_head"], hook: HookPoint) -> None:
        """
        Cache the input to $W_O$.
        """
        self.cache_z = z

    def list_fwd_hooks(self) -> list[tuple[str, Callable]]:
        """
        Returns a list of hook_point names and functions to call for the forward pass of
        the model using LoRA.
        """
        fwd_hooks = []
        # Attention Hooks qkv
        fwd_hooks.append((f"blocks.{self.layer_idx}.ln1.hook_normalized", self.store_hook_attn_normalized))
        fwd_hooks.append((f"blocks.{self.layer_idx}.attn.hook_q", self.lora_hook_qkv))
        fwd_hooks.append((f"blocks.{self.layer_idx}.attn.hook_k", self.lora_hook_qkv))
        fwd_hooks.append((f"blocks.{self.layer_idx}.attn.hook_v", self.lora_hook_qkv))
        # Attention Hooks z/out
        fwd_hooks.append((f"blocks.{self.layer_idx}.attn.hook_z", self.store_hook_z))
        fwd_hooks.append((f"blocks.{self.layer_idx}.hook_attn_out", self.lora_hook_out))

        return fwd_hooks

    def lora_hook_qkv(
        self, qkv_hook_out: Float[Tensor, "batch pos n_heads d_head"], hook: HookPoint
    ) -> Float[Tensor, "batch pos n_heads d_head"]:
        """
        Applies the LoRA modules to query/key/value, based on the hook location.
        Args:
            hook_qkv_out: Float[Tensor, "batch pos n_heads d_head"]
                The original output from query/key/value.
            hook: HookPoint
        Returns:
            The original output from query/key/value, plus the output from the corresponding LoRA module.
        """

        raise NotImplementedError()

    def lora_hook_out(
        self, attn_out: Float[Tensor, "batch pos n_heads d_head"], hook: HookPoint
    ) -> Float[Tensor, "batch pos n_heads d_head"]:
        """
        Applies the LoRA modules to the output projection matrix W_O in the attention layer.
        The output of the LoRA module is computed per head, so we sum over heads before adding
        to the activation `attn_out`.

        Args:
            attn_out: Float[Tensor, "batch pos n_heads d_head"]
                The output from the attention layer.
            hook: HookPoint
        Returns:
            The original output from the attention layer, plus the output from the LoRA module.
        """

        raise NotImplementedError()

In [ ]:
tests_lora.testing_lora_hooks(LoraHooks)
tests_lora.testing_lora_hooks_qkv_dispatch_and_out(LoraHooks)
print("All tests for LoraHooks passed!")

<details><summary>Solution</summary>

```python
class LoraHooks(nn.Module):
    """
    Defines the LoRA hooks needed for the Attention Layers of the transformer.
    (Could be modified to add LoRA to the MLP layers)
    """

    lora_q: Lora
    lora_k: Lora
    lora_v: Lora
    lora_o: Lora
    cache_qkv_in: Float[Tensor, "batch pos d_model"] = None
    cache_z: Float[Tensor, "batch pos n_heads d_head"] = None

    def __init__(
        self,
        layer_idx: int,
        cfg: HookedTransformerConfig,
        lora_alpha: float = 32,
        rank: int = 4,
        dtype: t.dtype = None,
    ):
        super().__init__()
        self.layer_idx = layer_idx
        self.rank = rank
        self.lora_alpha = lora_alpha
        self.dtype = dtype

        self.n_qo_heads = n_qo_heads = cfg.n_heads
        self.n_kv_heads = n_kv_heads = cfg.n_key_value_heads if cfg.n_key_value_heads is not None else cfg.n_heads
        d_model, d_head = cfg.d_model, cfg.d_head

        self.lora_q = Lora(d_model, d_head, n_inst=n_qo_heads, rank=rank, lora_alpha=lora_alpha, dtype=dtype)
        self.lora_k = Lora(d_model, d_head, n_inst=n_kv_heads, rank=rank, lora_alpha=lora_alpha, dtype=dtype)
        self.lora_v = Lora(d_model, d_head, n_inst=n_kv_heads, rank=rank, lora_alpha=lora_alpha, dtype=dtype)
        self.lora_o = Lora(d_head, d_model, n_inst=n_qo_heads, rank=rank, lora_alpha=lora_alpha, dtype=dtype)

    def store_hook_attn_normalized(self, normalized: Float[Tensor, "batch pos d_model"], hook: HookPoint) -> None:
        """
        Cache the input to query/key/value.
        """
        self.cache_qkv_in = normalized

    def store_hook_z(self, z: Float[Tensor, "batch pos n_heads d_head"], hook: HookPoint) -> None:
        """
        Cache the input to $W_O$.
        """
        self.cache_z = z

    def list_fwd_hooks(self) -> list[tuple[str, Callable]]:
        """
        Returns a list of hook_point names and functions to call for the forward pass of
        the model using LoRA.
        """
        fwd_hooks = []
        # Attention Hooks qkv
        fwd_hooks.append((f"blocks.{self.layer_idx}.ln1.hook_normalized", self.store_hook_attn_normalized))
        fwd_hooks.append((f"blocks.{self.layer_idx}.attn.hook_q", self.lora_hook_qkv))
        fwd_hooks.append((f"blocks.{self.layer_idx}.attn.hook_k", self.lora_hook_qkv))
        fwd_hooks.append((f"blocks.{self.layer_idx}.attn.hook_v", self.lora_hook_qkv))
        # Attention Hooks z/out
        fwd_hooks.append((f"blocks.{self.layer_idx}.attn.hook_z", self.store_hook_z))
        fwd_hooks.append((f"blocks.{self.layer_idx}.hook_attn_out", self.lora_hook_out))

        return fwd_hooks

    def lora_hook_qkv(
        self, qkv_hook_out: Float[Tensor, "batch pos n_heads d_head"], hook: HookPoint
    ) -> Float[Tensor, "batch pos n_heads d_head"]:
        """
        Applies the LoRA modules to query/key/value, based on the hook location.
        Args:
            hook_qkv_out: Float[Tensor, "batch pos n_heads d_head"]
                The original output from query/key/value.
            hook: HookPoint
        Returns:
            The original output from query/key/value, plus the output from the corresponding LoRA module.
        """

        hook_location = hook.name.split(".")[-1]

        qkv_in = self.cache_qkv_in
        qkv_in_repeated = einops.repeat(qkv_in, "batch pos d_model -> batch pos n_inst d_model", n_inst=1)

        if hook_location == "hook_q":
            return qkv_hook_out + self.lora_q(qkv_in_repeated)
        elif hook_location == "hook_k":
            return qkv_hook_out + self.lora_k(qkv_in_repeated)
        elif hook_location == "hook_v":
            return qkv_hook_out + self.lora_v(qkv_in_repeated)
        else:
            raise ValueError(f"Invalid hook location: {hook_location}")

    def lora_hook_out(
        self, attn_out: Float[Tensor, "batch pos n_heads d_head"], hook: HookPoint
    ) -> Float[Tensor, "batch pos n_heads d_head"]:
        """
        Applies the LoRA modules to the output projection matrix W_O in the attention layer.
        The output of the LoRA module is computed per head, so we sum over heads before adding
        to the activation `attn_out`.

        Args:
            attn_out: Float[Tensor, "batch pos n_heads d_head"]
                The output from the attention layer.
            hook: HookPoint
        Returns:
            The original output from the attention layer, plus the output from the LoRA module.
        """

        lora_result = self.lora_o(self.cache_z)
        lora_attn_out = einops.einsum(lora_result, "... n_heads d_model -> ... d_model")
        return attn_out + lora_attn_out
```
</details>

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

def _dd_report_complete():
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    try:
        body = _dd_json.dumps({
            'exercise_id': DD_EXERCISE_ID,
            'passed': True,
        }).encode('utf-8')
        req = _dd_req.Request(
            f'{DD_BACKEND_URL}/api/arena/complete',
            data=body,
            headers={
                'Content-Type': 'application/json',
                'Authorization': f'Bearer {DD_TOKEN}',
            },
            method='POST',
        )
        with _dd_req.urlopen(req, timeout=3) as r:
            r.read()
        print(f'[Delta Drills] reported completion of {DD_EXERCISE_ID}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

# This exercise has no automatic test — call `_dd_report_complete()`
# in a new cell once you're satisfied with your answer.
